# AHC VAD — Embedding Cache + Head Training (Colab, T4)

Data comes from **ZIP files downloaded with `gdown --fuzzy`** into `/content` (no Drive mount).

Layouts differ and are auto-detected:
- **flat** (test): `<root>/videos/*.mp4` + `<root>/ground_truth.csv`
- **per_class** (train): `<root>/<class>/videos/*.mp4` + `<root>/<class>/ground_truth.csv`

**Workflow:** set `DATASET = "test"`, Runtime -> Run all (smoke test). When it finishes clean,
set `DATASET = "train"` and Run all again. Caches and heads are written per-dataset, so the two
runs never clobber each other.

A CPU-only **pre-flight** validates the archive (layout, columns, video_id overlap) *before* any
GPU work, so a bad layout fails in seconds instead of after the whole embedding run.

T4-safe: fp16, SDPA kernels (flash disabled), no bf16, no flash-attention.

In [ ]:
!pip -q install open_clip_torch opencv-python-headless

import torch
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
print("CUDA:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Config — flip `DATASET` between runs

In [ ]:
import os

DATASET     = "test"                      # "test" for the smoke run, then "train"
ZIP_PATH    = f"/content/{DATASET}.zip"
EXTRACT_DIR = f"/content/{DATASET}_extracted"
CACHE_DIR   = f"/content/emb_cache_{DATASET}"
HEAD_PATH   = f"/content/head_{DATASET}.pt"

FPS_SAMPLE  = 2.0
DIFF_THRESH = 8.0

CLASSES = [
    "normal", "traffic_accident", "traffic_congestion", "stalled_or_broken_down_vehicle",
    "vehicle_blocking_traffic", "wrong_way_driving", "road_spill_or_debris",
    "waterlogging_or_flood", "fire", "smoke", "fighting_or_violence",
    "loitering_or_suspicious_presence",
]
IDX = {c: i for i, c in enumerate(CLASSES)}

assert os.path.exists(ZIP_PATH), f"archive not found: {ZIP_PATH} (download it with gdown --fuzzy first)"
print("dataset:", DATASET, "| zip:", ZIP_PATH)

## 1. Unzip, inspect layout, and pre-flight validate (CPU only)

Extracts `ZIP_PATH`, locates the root, auto-detects the layout, prints the tree, then checks
ground-truth columns and video_id overlap. Any problem raises here, before the GPU is touched.

In [ ]:
import zipfile, glob, csv


def has_flat(root):
    return (os.path.isfile(os.path.join(root, "ground_truth.csv"))
            and os.path.isdir(os.path.join(root, "videos")))


def has_per_class(root):
    for d in os.listdir(root):
        p = os.path.join(root, d)
        if os.path.isdir(p) and os.path.isdir(os.path.join(p, "videos")):
            return True
    return False


def detect_root(base, dataset):
    for c in [os.path.join(base, dataset), base]:
        if os.path.isdir(c) and (has_flat(c) or has_per_class(c)):
            return c
    for d, _, _ in os.walk(base):
        if has_flat(d) or has_per_class(d):
            return d
    return None


def detect_layout(root):
    if has_flat(root):
        return "flat"
    if has_per_class(root):
        return "per_class"
    return None


def print_tree(root, max_entries=8, max_depth=3):
    root = root.rstrip("/")
    base_depth = root.count(os.sep)
    for cur, dirs, files in os.walk(root):
        depth = cur.count(os.sep) - base_depth
        if depth > max_depth:
            dirs[:] = []
            continue
        dirs.sort()
        indent = "  " * depth
        print(f"{indent}{os.path.basename(cur) or cur}/")
        for fn in sorted(files)[:max_entries]:
            print(f"{indent}  {fn}")
        if len(files) > max_entries:
            print(f"{indent}  ... (+{len(files) - max_entries} more files)")
        if len(dirs) > max_entries:
            print(f"{indent}  ... (+{len(dirs) - max_entries} more dirs)")
            dirs[:] = dirs[:max_entries]


def video_paths(root, layout):
    pats = ([os.path.join(root, "videos", "*.mp4")] if layout == "flat"
            else [os.path.join(root, "*", "videos", "*.mp4")])
    paths = sorted(p for pat in pats for p in glob.glob(pat))
    if not paths:
        raise RuntimeError(f"no videos found. layout={layout}, searched: {pats}")
    return paths


def gt_csv_paths(root, layout):
    if layout == "flat":
        return [os.path.join(root, "ground_truth.csv")]
    return sorted(glob.glob(os.path.join(root, "*", "ground_truth.csv")))


def load_all_gt(root, layout):
    gt = {}
    csvs = gt_csv_paths(root, layout)
    for csv_path in csvs:
        if not os.path.isfile(csv_path):
            continue
        with open(csv_path, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            if not reader.fieldnames or "video_id" not in reader.fieldnames:
                raise RuntimeError(f"{csv_path}: missing 'video_id' column; found {reader.fieldnames}")
            for row in reader:
                vid = (row.get("video_id") or "").strip()
                if not vid:
                    continue
                cls = (row.get("class_name") or "normal").strip()
                s = str(row.get("start_time_sec", "") or "").strip()
                e = str(row.get("end_time_sec", "") or "").strip()
                gt.setdefault(vid, []).append((cls, float(s) if s else None, float(e) if e else None))
    if not gt:
        raise RuntimeError(f"no ground_truth rows loaded; searched: {csvs}")
    return gt


ROOT = detect_root(EXTRACT_DIR, DATASET)
if ROOT is None:
    print("extracting", ZIP_PATH, "->", EXTRACT_DIR)
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)
    ROOT = detect_root(EXTRACT_DIR, DATASET)
if ROOT is None:
    raise RuntimeError(f"could not locate a dataset root under {EXTRACT_DIR} after extracting {ZIP_PATH}")

LAYOUT = detect_layout(ROOT)
if LAYOUT is None:
    raise RuntimeError(f"unrecognized layout at {ROOT}: expected flat (videos/ + ground_truth.csv) "
                       f"or per_class (<class>/videos/ + <class>/ground_truth.csv)")

print("ROOT   :", ROOT)
print("LAYOUT :", LAYOUT)
print("-" * 52)
print_tree(ROOT)

PATHS = video_paths(ROOT, LAYOUT)
GT = load_all_gt(ROOT, LAYOUT)
vid_ids = {os.path.splitext(os.path.basename(p))[0] for p in PATHS}
gt_ids = set(GT)
overlap = vid_ids & gt_ids
if not overlap:
    raise RuntimeError("no video_id overlap between videos and ground_truth. "
                       f"{len(vid_ids)} videos e.g. {sorted(vid_ids)[:3]}; "
                       f"{len(gt_ids)} gt ids e.g. {sorted(gt_ids)[:3]}")
unknown = sorted({c for spans in GT.values() for (c, _, _) in spans if c != "normal" and c not in IDX})

print("-" * 52)
print("PRE-FLIGHT OK")
print("  videos            :", len(vid_ids))
print("  gt videos         :", len(gt_ids))
print("  video_id overlap  :", len(overlap))
print("  videos without gt :", len(vid_ids - gt_ids))
print("  gt without video  :", len(gt_ids - vid_ids))
if unknown:
    print("  WARNING unknown classes (will be labeled 'normal'):", unknown)

## 2. Embedding cache — 2 fps, drop near-static frames, ViT-B-16-SigLIP

Writes one `.npz` per video (`emb` L2-normalized, `times`, `duration`). Re-runnable: already-cached
videos are skipped. Input size for the shape probe is taken from `preprocess`, not hardcoded.

In [ ]:
import numpy as np
import cv2
from PIL import Image
import open_clip
import time

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-16-SigLIP", pretrained="webli", device=DEVICE)
model = model.to(DTYPE).eval()

with torch.no_grad():
    _px = preprocess(Image.new("RGB", (256, 256))).unsqueeze(0).to(DEVICE, DTYPE)
    EMB_DIM = int(model.encode_image(_px).shape[-1])
print("embedding dim:", EMB_DIM)


def sample_frames(path, fps_sample=FPS_SAMPLE, diff_thresh=DIFF_THRESH):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    step = max(1, int(round(fps / fps_sample)))
    frames, times, idx, prev = [], [], 0, None
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % step == 0:
            small = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (64, 64)).astype(np.float32)
            if prev is None or np.abs(small - prev).mean() > diff_thresh:
                frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
                times.append(idx / fps)
                prev = small
        idx += 1
    cap.release()
    return frames, times, (idx / fps if fps else 0.0)


@torch.no_grad()
def embed_frames(frames, batch=64):
    out = []
    for i in range(0, len(frames), batch):
        px = torch.stack([preprocess(f) for f in frames[i:i+batch]]).to(DEVICE, DTYPE)
        feat = model.encode_image(px)
        feat = feat / feat.norm(dim=-1, keepdim=True)
        out.append(feat.float().cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, EMB_DIM), dtype=np.float32)


os.makedirs(CACHE_DIR, exist_ok=True)
print("videos to process:", len(PATHS))

t0 = time.time()
total_new = 0
for n, path in enumerate(PATHS, 1):
    vid = os.path.splitext(os.path.basename(path))[0]
    dst = os.path.join(CACHE_DIR, vid + ".npz")
    if os.path.exists(dst):
        continue
    frames, times, dur = sample_frames(path)
    emb = embed_frames(frames)
    np.savez_compressed(dst, emb=emb, times=np.array(times, dtype=np.float32), duration=np.float32(dur))
    total_new += len(frames)
    if n % 25 == 0 or n == len(PATHS):
        print(f"[{n}/{len(PATHS)}] {vid}: {len(frames)} frames, {dur:.1f}s  ({time.time()-t0:.0f}s elapsed)")

cached = glob.glob(os.path.join(CACHE_DIR, "*.npz"))
if not cached:
    raise RuntimeError(f"embedding cache is empty at {CACHE_DIR} (searched videos under {ROOT}, layout={LAYOUT})")
print("cache dir:", CACHE_DIR, "| npz files:", len(cached), "| new frames this run:", total_new)

## 3. Train the classifier head — frame labels from event intervals, class-balanced loss

Frame label = the event class whose interval contains the frame time; whole-clip labels (null
timestamps) apply to every frame; the rest is `normal`. Batches are moved to the GPU one at a
time so a large training set can't OOM the card.

In [ ]:
import torch.nn as nn


def frame_label(spans, t):
    for cls, s, e in spans:
        if cls == "normal" or cls not in IDX:
            continue
        if s is None or e is None:
            return IDX[cls]
        if s <= t <= e:
            return IDX[cls]
    return IDX["normal"]


def build_dataset(cache_dir, gt):
    X, y = [], []
    for npz in sorted(glob.glob(os.path.join(cache_dir, "*.npz"))):
        vid = os.path.splitext(os.path.basename(npz))[0]
        if vid not in gt:
            continue
        d = np.load(npz)
        emb, times = d["emb"], d["times"]
        spans = gt[vid]
        for k in range(len(emb)):
            X.append(emb[k])
            y.append(frame_label(spans, float(times[k])))
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.int64)


X, y = build_dataset(CACHE_DIR, GT)
if X.shape[0] == 0:
    raise RuntimeError(f"zero labeled frames: {len(GT)} gt videos vs cache {CACHE_DIR}. "
                       f"video_ids likely mismatch between cache filenames and ground_truth.csv")
print("dataset:", X.shape, "labels:", y.shape)


class Head(nn.Module):
    def __init__(self, dim, n):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, 512), nn.GELU(), nn.Dropout(0.3), nn.Linear(512, n))
    def forward(self, x):
        return self.net(x)


counts = np.bincount(y, minlength=len(CLASSES))
w = 1.0 / np.clip(counts, 1, None)
w = w / w.sum() * len(CLASSES)

Xt = torch.from_numpy(X)
yt = torch.from_numpy(y)
head = Head(X.shape[1], len(CLASSES)).to(DEVICE)
opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
lossf = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=DEVICE))

EPOCHS, BS = 40, 4096
n = X.shape[0]
for ep in range(EPOCHS):
    head.train()
    perm = torch.randperm(n)
    tot = 0.0
    for i in range(0, n, BS):
        b = perm[i:i+BS]
        xb = Xt[b].to(DEVICE)
        yb = yt[b].to(DEVICE)
        opt.zero_grad()
        loss = lossf(head(xb), yb)
        loss.backward()
        opt.step()
        tot += loss.item() * b.numel()
    if (ep + 1) % 10 == 0 or ep == 0:
        print(f"epoch {ep+1}/{EPOCHS}  loss {tot/n:.4f}")

torch.save({"state": head.state_dict(), "dim": X.shape[1], "classes": CLASSES,
            "backbone": "ViT-B-16-SigLIP", "pretrained": "webli"}, HEAD_PATH)
print("saved:", HEAD_PATH)

## 4. Report

In [ ]:
cached = sorted(glob.glob(os.path.join(CACHE_DIR, "*.npz")))
total_frames = int(counts.sum())

print("=" * 52)
print(f"REPORT  ({DATASET}, layout={LAYOUT})")
print("=" * 52)
print(f"videos cached       : {len(cached)}")
print(f"total frames cached : {total_frames}")
print("\nper-class frame counts:")
for c, ct in zip(CLASSES, counts.tolist()):
    print(f"  {c:<34} {ct:>8}")
print("\nfiles:")
print(f"  embedding cache : {CACHE_DIR}/  ({len(cached)} .npz)")
print(f"  trained head    : {HEAD_PATH}")
print("\nnext: set DATASET = 'train' in the config cell and Run all again.")